In [ ]:

# Лабораторная работа № 4.2: Наследование, Полиморфизм и Инкапсуляция
# Вариант 9: Маркетинг


In [1]:
"""
Лабораторная работа № 4.2: Наследование, Полиморфизм и Инкапсуляция
Вариант 9: Маркетинг
"""

from typing import List


# ==========================================
# ИСКЛЮЧЕНИЯ (Иерархия ошибок)
# ==========================================

class MarketingError(Exception):
    """Базовый класс для всех ошибок маркетинговой системы"""
    pass


class InvalidBudgetError(MarketingError):
    """Ошибка при некорректном бюджете кампании"""
    pass


class InvalidRevenueError(MarketingError):
    """Ошибка при некорректной выручке"""
    pass


class InvalidClientError(MarketingError):
    """Ошибка при некорректных данных клиента"""
    pass


# ==========================================
# БАЗОВЫЕ КЛАССЫ
# ==========================================

class Campaign:
    """БК1: Базовый класс маркетинговой кампании"""
    def __init__(self, name: str, budget: float, revenue: float):

        if not isinstance(name, str) or not name.strip():
            raise ValueError("Название кампании не может быть пустым")
        if budget <= 0:
            raise InvalidBudgetError(
                f"Бюджет кампании '{name}' должен быть > 0, получено: {budget}"
            )
        if revenue < 0:
            raise InvalidRevenueError(
                f"Выручка кампании '{name}' не может быть отрицательной, получено: {revenue}"
            )

        self.name = name
        self.budget = budget
        self.revenue = revenue

    def calculate_roi(self) -> float:
        """Базовый расчет ROI (Return on Investment) в процентах"""
        try:
            return ((self.revenue - self.budget) / self.budget) * 100
        except ZeroDivisionError:

            raise InvalidBudgetError(f"Невозможно рассчитать ROI: бюджет равен 0")


class Client:
    """БК2: Базовый класс клиента"""
    def __init__(self, name: str, contact_email: str):
        if not isinstance(name, str) or not name.strip():
            raise InvalidClientError("Имя клиента не может быть пустым")
        if "@" not in contact_email or "." not in contact_email:
            raise InvalidClientError(f"Некорректный email: {contact_email}")

        self.name = name
        self.contact_email = contact_email

    def get_client_type(self) -> str:
        return "Обычный клиент"


# ==========================================
# ДОЧЕРНИЕ КЛАССЫ 1 (Специализация кампаний)
# ==========================================

class SocialMediaCampaign(Campaign):
    """ДК1.1: SMM-кампания"""
    VALID_PLATFORMS = ('TikTok', 'Instagram', 'Telegram', 'VK', 'Facebook', 'Twitter')

    def __init__(self, name: str, budget: float, revenue: float, platform: str):
        super().__init__(name, budget, revenue)
        if platform not in self.VALID_PLATFORMS:
            raise ValueError(
                f"Неизвестная платформа '{platform}'. "
                f"Допустимые: {', '.join(self.VALID_PLATFORMS)}"
            )
        self.platform = platform

    def calculate_roi(self) -> float:
        """Полиморфизм: ROI корректируется коэффициентом платформы"""
        base_roi = super().calculate_roi()
        multipliers = {
            'TikTok': 1.25,
            'Instagram': 1.15,
            'Telegram': 1.10,
            'VK': 1.05,
            'Facebook': 0.95,
            'Twitter': 0.90
        }
        return base_roi * multipliers.get(self.platform, 1.0)


class SEOCampaign(Campaign):
    """ДК1.2: SEO-кампания"""
    def __init__(self, name: str, budget: float, revenue: float, keywords_count: int):
        super().__init__(name, budget, revenue)
        if not isinstance(keywords_count, int) or keywords_count < 0:
            raise ValueError(
                f"Количество ключевых слов должно быть >= 0, получено: {keywords_count}"
            )
        self.keywords_count = keywords_count

    def calculate_roi(self) -> float:
        """Полиморфизм: ROI увеличивается на 0.5% за каждое ключевое слово"""
        base_roi = super().calculate_roi()
        return base_roi + self.keywords_count * 0.5


# ==========================================
# ДОЧЕРНИЙ КЛАСС 2 (Специализация клиента)
# ==========================================

class LongTermClient(Client):
    """ДК2: Долгосрочный клиент с ежемесячным платежом"""
    def __init__(self, name: str, contact_email: str,
                 monthly_payment: float, months_active: int):
        super().__init__(name, contact_email)
        if monthly_payment <= 0:
            raise InvalidClientError(
                f"Ежемесячный платёж должен быть > 0, получено: {monthly_payment}"
            )
        if not isinstance(months_active, int) or months_active <= 0:
            raise InvalidClientError(
                f"Срок сотрудничества должен быть целым числом > 0, "
                f"получено: {months_active}"
            )
        self.monthly_payment = monthly_payment
        self.months_active = months_active

    def get_client_type(self) -> str:
        return f"LongTermClient (стаж: {self.months_active} мес.)"

    def get_loyalty_bonus(self) -> float:
        """Возвращает бонус к итоговому ROI за лояльность (макс. 5%)"""
        bonus = self.months_active * 0.2
        return min(bonus, 5.0)


# ==========================================
# КЛАСС-КОНТЕЙНЕР (Композиция)
# ==========================================

class MarketingPlan:
    """Контейнер: Управляет клиентом и списком кампаний"""
    def __init__(self, client: Client):
        if not isinstance(client, Client):
            raise TypeError(
                f"Ожидался объект Client, получено: {type(client).__name__}"
            )
        self.client = client
        self.campaigns: List[Campaign] = []

    def add_campaign(self, campaign: Campaign) -> None:
        """Добавляет кампанию в план с проверкой типа"""
        if not isinstance(campaign, Campaign):
            raise TypeError(
                f"Можно добавить только объект Campaign, "
                f"получено: {type(campaign).__name__}"
            )
        self.campaigns.append(campaign)

    def generate_report(self) -> str:
        """Генерация детализированного отчёта"""
        if not self.campaigns:
            raise MarketingError(
                f"Невозможно сформировать отчёт для '{self.client.name}': "
                f"список кампаний пуст"
            )

        report_lines = [
            "=" * 70,
            "ОТЧЁТ ПО МАРКЕТИНГОВОМУ ПЛАНУ",
            "=" * 70,
            f"Клиент: {self.client.name} ({self.client.get_client_type()})",
            f"Контакт: {self.client.contact_email}",
            "-" * 70,
            f"{'№':<3} | {'Название кампании':<35} | {'Тип':<5} | {'ROI, %':<8}",
            "-" * 70
        ]

        total_roi_sum = 0.0

        for i, campaign in enumerate(self.campaigns, 1):
            try:
                roi = campaign.calculate_roi()
            except MarketingError as e:

                report_lines.append(
                    f"{i:<3} | {campaign.name:<35} | ОШИБКА: {e}"
                )
                continue

            total_roi_sum += roi
            camp_type = "SMM" if isinstance(campaign, SocialMediaCampaign) else "SEO"
            report_lines.append(
                f"{i:<3} | {campaign.name:<35} | {camp_type:<5} | {roi:>7.2f}%"
            )

        avg_roi = total_roi_sum / len(self.campaigns)


        client_bonus = 0.0
        if isinstance(self.client, LongTermClient):
            client_bonus = self.client.get_loyalty_bonus()
            report_lines.append("-" * 70)
            report_lines.append(
                f"Бонус лояльности клиента: +{client_bonus:.2f}% к среднему ROI"
            )

        final_roi = avg_roi + client_bonus

        report_lines.extend([
            "=" * 70,
            f"Средний ROI кампаний: {avg_roi:>36.2f}%",
            f"ИТОГОВЫЙ ROI с учетом бонусов: {final_roi:>20.2f}%",
            "=" * 70
        ])

        return "\n".join(report_lines)


# ==========================================
# ДЕМО-СЦЕНАРИЙ (main) с try-except
# ==========================================

if __name__ == "__main__":
    print("Запуск демонстрации бизнес-процесса (Вариант 9: Маркетинг)\n")
    print("=" * 70)
    print("ЧАСТЬ 1: Демонстрация обработки исключений")
    print("=" * 70)


    print("\n[Тест 1] Попытка создать кампанию с нулевым бюджетом:")
    try:
        bad_campaign = Campaign("Провал", budget=0, revenue=1000)
    except InvalidBudgetError as e:
        print(f"Исключение перехвачено: {e}")
    except MarketingError as e:
        print(f"Базовое исключение: {e}")

    print("\n[Тест 2] Попытка создать кампанию с отрицательной выручкой:")
    try:
        bad_campaign = Campaign("Убыток", budget=10000, revenue=-5000)
    except InvalidRevenueError as e:
        print(f"Исключение перехвачено: {e}")


    print("\n[Тест 3] Попытка создать клиента с невалидным email:")
    try:
        bad_client = Client("ООО 'Рога и Копыта'", "invalid-email")
    except InvalidClientError as e:
        print(f"Исключение перехвачено: {e}")


    print("\n[Тест 4] Попытка создать SMM-кампанию на неизвестной платформе:")
    try:
        bad_smm = SocialMediaCampaign(
            "Реклама", budget=10000, revenue=15000, platform="MySpace"
        )
    except ValueError as e:
        print(f"Исключение перехвачено: {e}")


    print("\n[Тест 5] Попытка добавить в план не-Campaign объект:")
    try:
        plan = MarketingPlan(Client("Тест", "test@test.ru"))
        plan.add_campaign("это строка, а не кампания")
    except TypeError as e:
        print(f"Исключение перехвачено: {e}")

    print("\n[Тест 6] Попытка сформировать отчёт без кампаний:")
    try:
        empty_plan = MarketingPlan(Client("Пусто", "empty@test.ru"))
        print(empty_plan.generate_report())
    except MarketingError as e:
        print(f"Исключение перехвачено: {e}")

    # ==========================================
    # УСПЕШНЫЙ СЦЕНАРИЙ
    # ==========================================
    print("\n\n")
    print("=" * 70)
    print("ЧАСТЬ 2: Успешный бизнес-сценарий")
    print("=" * 70)

    try:

        regular_client = Client("ООО 'Стартап'", "info@startup.ru")
        long_term_client = LongTermClient(
            name="ПАО 'Корпорация'",
            contact_email="marketing@corp.ru",
            monthly_payment=150000.0,
            months_active=18
        )


        smm_1 = SocialMediaCampaign(
            "Instagram Reels Запуск", 50000.0, 85000.0, "Instagram"
        )
        smm_2 = SocialMediaCampaign(
            "TikTok Вирусный ролик", 30000.0, 60000.0, "TikTok"
        )
        seo_1 = SEOCampaign(
            "SEO 'купить слона'", 40000.0, 70000.0, 15
        )


        plan_regular = MarketingPlan(regular_client)
        plan_regular.add_campaign(smm_1)
        plan_regular.add_campaign(seo_1)

        plan_long_term = MarketingPlan(long_term_client)
        plan_long_term.add_campaign(smm_1)
        plan_long_term.add_campaign(smm_2)
        plan_long_term.add_campaign(seo_1)


        print("\n")
        print(plan_regular.generate_report())
        print("\n\n")
        print(plan_long_term.generate_report())

    except MarketingError as e:
        print(f"\n Критическая ошибка бизнес-логики: {e}")
    except Exception as e:
        print(f"\n Непредвиденная ошибка: {type(e).__name__}: {e}")

Запуск демонстрации бизнес-процесса (Вариант 9: Маркетинг)

ЧАСТЬ 1: Демонстрация обработки исключений

[Тест 1] Попытка создать кампанию с нулевым бюджетом:
Исключение перехвачено: Бюджет кампании 'Провал' должен быть > 0, получено: 0

[Тест 2] Попытка создать кампанию с отрицательной выручкой:
Исключение перехвачено: Выручка кампании 'Убыток' не может быть отрицательной, получено: -5000

[Тест 3] Попытка создать клиента с невалидным email:
Исключение перехвачено: Некорректный email: invalid-email

[Тест 4] Попытка создать SMM-кампанию на неизвестной платформе:
Исключение перехвачено: Неизвестная платформа 'MySpace'. Допустимые: TikTok, Instagram, Telegram, VK, Facebook, Twitter

[Тест 5] Попытка добавить в план не-Campaign объект:
Исключение перехвачено: Можно добавить только объект Campaign, получено: str

[Тест 6] Попытка сформировать отчёт без кампаний:
Исключение перехвачено: Невозможно сформировать отчёт для 'Пусто': список кампаний пуст



ЧАСТЬ 2: Успешный бизнес-сценарий


ОТ